In [1]:
import time
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.callbacks import BaseCallbackHandler

# خواندن متغیرهای .env
load_dotenv()
model_name = os.getenv("OLLAMA_MODEL")
if not model_name:
    raise ValueError("OLLAMA_MODEL در فایل .env تعریف نشده است.")


# ============================================================
# 1. اطلاعات Request
# ============================================================

REQUEST_ID = "92831"
USER_ID = 125


# ============================================================
# 2. Callback
# ============================================================

class MonitoringCallback(BaseCallbackHandler):

    def __init__(self):
        super().__init__()

        # زمان شروع هر LLM
        self.llm_start_times = {}

        # زمان شروع هر Tool
        self.tool_start_times = {}


    # --------------------------------------------------------
    # LLM START
    # --------------------------------------------------------

    def on_llm_start(
        self,
        serialized,
        prompts,
        run_id,
        **kwargs
    ):

        self.llm_start_times[run_id] = time.perf_counter()

        print("\nLLM START")

        print(f"Model: {model_name} - RunId: {run_id}")


    # --------------------------------------------------------
    # LLM END
    # --------------------------------------------------------

    def on_llm_end(
        self,
        response,
        run_id,
        **kwargs
    ):

        start_time = self.llm_start_times.pop(
            run_id,
            None
        )

        if start_time is not None:

            duration = (
                time.perf_counter() - start_time
            )

            print(
                f"LLM END - RunId: {run_id}"
            )

            print(
                f"Duration: {duration:.2f}s"
            )

        else:

            print("LLM END")


    # --------------------------------------------------------
    # TOOL START
    # --------------------------------------------------------

    def on_tool_start(
        self,
        serialized,
        input_str,
        run_id,
        **kwargs
    ):

        self.tool_start_times[run_id] = (
            time.perf_counter()
        )

        tool_name = serialized.get(
            "name",
            "unknown"
        )

        print("\nTOOL START")

        print(
            f"Tool: {tool_name} - Inputs: {input_str}"
        )


    # --------------------------------------------------------
    # TOOL END
    # --------------------------------------------------------

    def on_tool_end(
        self,
        output,
        run_id,
        **kwargs
    ):

        start_time = self.tool_start_times.pop(
            run_id,
            None
        )

        if start_time is not None:

            duration = (
                time.perf_counter() - start_time
            )

            print("TOOL END")

            print(
                f"Duration: {duration:.2f}s"
            )

        else:

            print("TOOL END")


# ============================================================
# 3. Tool
# ============================================================

@tool
def add_numbers(a: int, b: int) -> int:
    """
    دو عدد را با هم جمع می‌کند.
    """

    return a + b


# ============================================================
# 4. Model
# ============================================================

model = init_chat_model(
    model=model_name,
    model_provider="ollama",
    temperature=0,
)


# ============================================================
# 5. Agent
# ============================================================

agent = create_agent(
    model=model,
    tools=[add_numbers],

    system_prompt="""
تو یک دستیار ساده هستی.

اگر کاربر از تو خواست دو عدد را جمع کنی،
حتماً از ابزار add_numbers استفاده کن.

خودت محاسبه را انجام نده.

بعد از دریافت نتیجه ابزار،
پاسخ نهایی را به فارسی بده.
""",
)


# ============================================================
# 6. اجرای Agent
# ============================================================

def run_agent(question: str):

    total_start = time.perf_counter()

    print("=" * 60)

    print(
        f"Request ID: {REQUEST_ID}"
    )

    print(
        f"User ID: {USER_ID}"
    )

    print()

    print("Agent START")

    print("=" * 60)


    # --------------------------------------------------------
    # Callback
    # --------------------------------------------------------

    callback = MonitoringCallback()


    # --------------------------------------------------------
    # Agent
    # --------------------------------------------------------

    result = agent.invoke(

        {
            "messages": [
                {
                    "role": "user",
                    "content": question,
                }
            ]
        },

        config={
            "callbacks": [
                callback
            ]
        }
    )


    # --------------------------------------------------------
    # Agent END
    # --------------------------------------------------------

    total_duration = (
        time.perf_counter()
        - total_start
    )


    print("\nAgent END")

    print(
        f"Total: {total_duration:.2f}s"
    )


    # --------------------------------------------------------
    # Final Answer
    # --------------------------------------------------------

    final_message = result["messages"][-1]

    print("\n" + "=" * 60)

    print("FINAL ANSWER")

    print("=" * 60)

    print(
        final_message.content
    )


# ============================================================
# 7. RUN
# ============================================================

run_agent(question=" دو عدد 17 و 23 را با هم جمع کن")

Request ID: 92831
User ID: 125

Agent START

LLM START
Model: hf.co/empero-ai/Qwen3.8-4B-Distill-GGUF:Q4_K_M - RunId: 01a0a366-b387-7c11-aaa2-4a749b999669
LLM END - RunId: 01a0a366-b387-7c11-aaa2-4a749b999669
Duration: 10.03s

TOOL START
Tool: add_numbers - Inputs: {'a': 17, 'b': 23}
TOOL END
Duration: 0.00s

LLM START
Model: hf.co/empero-ai/Qwen3.8-4B-Distill-GGUF:Q4_K_M - RunId: 01a0a366-dab7-7c20-9383-96ed30a91eeb
LLM END - RunId: 01a0a366-dab7-7c20-9383-96ed30a91eeb
Duration: 4.21s

Agent END
Total: 14.25s

FINAL ANSWER
جمع دو عدد ۱۷ و ۲۳ برابر است با ۴۰.
